# Final Product Demo  
__Objective:__ Create materials for an overview of the project.   
1. Materials to be added to the README.md  
2. Materials for a potential slide deck.

## Packages and Data

In [24]:
# packages

## data wrangling
import numpy as np

## link project directory
from pathlib import Path
import sys
dir = str(Path(Path().cwd()).parents[0])
if dir not in sys.path:
    sys.path.append(dir)

## custom code
from src.data_gathering.scryfall_dataset import ScryfallDataset
from src.modeling.auto_tagger_multi_lab import ScryfallTaggerFromPretrained

In [17]:
# constants
from src.config import (
    TASK,
    MODEL_NAME, DATASET_SOURCE, DATASET_SIZE_N, TEST_SIZE_N, TAG_SIZE,
    OUTPUT_DIR
)

In [14]:
# helper functions
def _repo_path(config_path:str):
    """
    Convert notebook-relative config paths such as ../data/... into
    repo-root absolute paths for script execution.
    """
    return str((dir / config_path.replace('../', '')).resolve())

In [16]:
# data
sf = ScryfallDataset(task = TASK)

if DATASET_SOURCE == 'build_from_scryfall':
    # build dataset as needed
    sf.build_dataset(
        card_path = '../data/oracle-cards.json',
        tag_path = '../reports/scryfall_tags.json',
        train_size_pct = 0.8,
        truncate_dataset = DATASET_SIZE_N,
        test_size_n = TEST_SIZE_N,
        top_n_tags = TAG_SIZE
    )

elif DATASET_SOURCE == 'load_from_scryfall':
    # assumes we have previously used DATASET_SOURCE == 'build_from_scryfall'
    sf.load_hf_dataset(
        train_path = f'../data/scryfall_{TASK}_train.json',
        val_path = f'../data/scryfall_{TASK}_val.json',
        test_path = f'../data/scryfall_{TASK}_test.json'
    )

elif DATASET_SOURCE == 'load_from_ocr':
    # assumes we have ran the scripts/generate_card_text_ocr_dataset.py
    sf.load_hf_dataset_ocr(
        filepath = '../data/card_image_ocr_text_tags.json'
    )

Scryfall OCR-To-Tag Multi Label Classification Dataset Loaded
	Train Records = 6713
	Val Records = 1680
	Test Records = 50
	Count Unique Tags = 300


## Load Model

In [19]:
# load the tagger
tagger = ScryfallTaggerFromPretrained(
    base_model_name = MODEL_NAME,
    n_labels = len(sf.unique_tags),
    output_dir = OUTPUT_DIR,
    id2label = sf.id2label,
    label2id = sf.label2id
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Demo The Model

### Test Set

In [42]:
samples = np.random.choice(sf.dataset['test'], size = 5)
for s in samples:
    print('=' * 20)
    header1 = f'----- Card Text OCR -----'
    print(f'*****\nCard Name = {s["card_name"]} (Oracle ID = {s["oracle_id"]})\n*****')
    print(f'\n{header1}\n{s["card_text_ocr"]}\n{"-" * len(header1)}\n')

    header2 = f'----- Tags -----'
    pred = tagger.generate_tags(
        card_text = s['card_text_ocr'],
        threshold = 0.7,
        top_k = 5
    )
    print(header2)
    print(f'True Tags = {sorted(s["tags"])}')
    print(f'Pred Tags = {sorted(pred)}')
    print(f'{"-" * len(header2)}\n\n')

*****
Card Name = Wrap in Vigor (Oracle ID = 39da2aa8-f4d9-44f6-a446-488beaec821f)
*****

----- Card Text OCR -----
Wrap in Vigor

Instant

Regenerate each creature you control.

Some nature mages unknowingly took advantage of the temporal energies still swirling on Dominaria. What they mistook for healing magic was in fact the manipulation of time.
-------------------------

----- Tags -----
True Tags = ['combat trick', 'protects-creature', 'regenerates other']
Pred Tags = ['protects-creature']
----------------


*****
Card Name = Jeska, Warrior Adept (Oracle ID = 3186fddd-23fd-440c-ad61-b4130e00f765)
*****

----- Card Text OCR -----
Jeska, Warrior Adept

Creature -- Barbarian Legend

First strike, haste

: Jeska, Warrior Adept deals 1 damage to target creature or player.

"My brother and I both come from Balthor's forge. Kamahl has a temper of fire. I have a temper of steel.
-------------------------

----- Tags -----
True Tags = ['activated ability', 'burn any', 'deprecated legend t

### Truly Unseen Data  
__The Hobbit Spoilers__

In [46]:
# generate text
from docling.document_converter import DocumentConverter

## get a list of files to generate text from
hob_folder = Path('../data/demo_images/hob')
hob_imgs = [Path(hob_folder / p.name) for p in hob_folder.iterdir() if p.is_file()]

## generate text
hob_res = {}
converter = DocumentConverter()
for img in hob_imgs:
    hob_res[img] = converter.convert(img).document.export_to_text()

[INFO] 2026-07-29 13:35:28,488 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-29 13:35:28,489 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-29 13:35:28,497 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nickcruickshank/Projects/mtg-multimodal-classification/.venv/lib/python3.14/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-29 13:35:28,497 [RapidOCR] main.py:50: Using /Users/nickcruickshank/Projects/mtg-multimodal-classification/.venv/lib/python3.14/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-29 13:35:28,772 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-29 13:35:28,772 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-29 13:35:28,774 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nickcruickshank/Projects/mtg-multimodal-classification/.venv/lib/python3.14/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-07-

In [50]:
# generate tags
for img, txt in hob_res.items():
    h1 = f'*** {img} ***'
    print(f'{"*" * len(h1)}\n{h1}\n{"*" * len(h1)}')

    h2 = f'----- OCR Generated Text -----'
    print(f'\n{h2}\n{txt}\n{"-" * len(h2)}\n')

    pred = tagger.generate_tags(card_text = txt, threshold = 0.7, top_k = 5)
    print(f'Predicted Tags = {pred}\n')



**************************************************************
*** ../data/demo_images/hob/hob-134-part-in-friendship.png ***
**************************************************************

----- OCR Generated Text -----
Part in Friendship

Enchantment

Whenever a nontoken creature you control dies, reveal cards from the top of your library until you reveal a creature e card. If its mana value is s less than or equal to the number of lands you control, put it onto the battlefield. Otherwise, put it into your hand. Put the rest on the bottom of your library in a random order. This ability triggers only once each turn.
------------------------------

Predicted Tags = ['death trigger', 'regrowth-creature', 'creaturefall', 'mana value matters']

****************************************************************
*** ../data/demo_images/hob/hob-169-tom-bert-and-william.png ***
****************************************************************

----- OCR Generated Text -----
Tom,Bert,and William
